# D1 MRI interpolation sensitivity: a short matched training study

Original D1 train/test animal assignments are preserved. MouseFull adds balanced D3 holdouts; rats keep all longitudinal scans of each animal together.

`USE_VALIDATION_FOR_SELECTION=False` trains the complete epoch budget without validation passes, uses a preset cosine learning-rate schedule, and evaluates the last epoch. Test animals are evaluated only after fitting.

Set it to `True` to reserve separate inner validation from the training pool and use ReduceLROnPlateau plus validation-selected weights. It never uses test animals for checkpoint selection. Start a fresh run when changing this policy.


In [13]:
from pathlib import Path
from dataclasses import dataclass, replace
import sys
import json

REPO_ROOT: Path = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src/models/networks.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import nibabel as nib
import numpy as np
import pandas as pd
import tensorflow as tf
import plotly.express as px
from IPython.display import display

from src.experiments.config import (
    DataConfig, DatasetSpec, EvaluationPatchConfig, ExperimentConfig, FixedSplitConfig,
    InferenceConfig, InitializationConfig, LabelConfig, LossConfig, ModelConfig,
    PostprocessingConfig, PreprocessingConfig,
)
from src.experiments.data import prepare_data, make_preprocessor
from src.experiments.ablation import study_selection, run_study
from src.experiments.training import configure_runtime
from src.evaluation.k_fold import TrainingConfig
from src.preprocessing.preprocessor import Preprocessor, Resample

REFERENCE: Path = REPO_ROOT / "example" / "RARE" / "TBI_fm_19_49" / "Anat" / "TBI_fm_19_49_N4.nii.gz"
ref_img = nib.load(REFERENCE)
print("Reference axes:", nib.aff2axcodes(ref_img.affine))
runtime = configure_runtime(seed=42, memory_growth=True, mixed_precision=False)

Reference axes: ('R', 'I', 'A')


## 1. Visible training budget and common configuration

Order 1 is the reference. N4 is already present in the input files. Intensity
blur/noise occur on the native grid, as in notebook 01; affine augmentation,
centre-biased random cropping (`mode="center", std=None`) and LR flipping remain
shared. Training merges the lateral ventricles; full-volume evaluation splits
them by centroid and restores original IDs.

An existing `SPLIT_FILE` can enforce exact memberships. The default comes from the selected
reference fold and is saved with the study. Do not reuse a final test cohort
for repeated tuning: use validation for choosing an interpolation setting. The
test comparison describes these four predeclared candidates.

The default `REFERENCE_SPLITS` is `DATA_ROOT.parent / "D1_splits_final.json"`.
The reference stores the original `train` and `test` animal lists.
Original D1 memberships are preserved without modality rebalancing;
optional inner validation comes only from training. Superseded files are backed up.
Fixed runs use `REFERENCE_FOLD=4`; K-Fold runs use the complete reference plan.
MouseFull adds balanced SHAM folds while preserving every updated D1 role.
Explicit `SPLIT_FILE`/`FOLD_FILE` and resumed memberships take precedence.
Set `REFERENCE_SPLITS=None` to generate a fresh plan. Mouse experiments start
from scratch on these memberships; weights are not shared across folds.


In [ ]:
DATA_ROOT: Path = Path("D:/Other files/TESI_CNN/MANUSCRIPT_DATASET/normalized")
REFERENCE_SPLITS: Path | None = DATA_ROOT.parent / "D1_splits_final.json"
REFERENCE_FOLD: int = 4  # Fixed studies reuse this fold; CV uses all reference folds.
STUDY_DIR: Path = REPO_ROOT / "results" / "interpolation" / f"d1_orders_0_1_2_reference_fold_{REFERENCE_FOLD}_balanced"
SPLIT_FILE: Path | None = None
RUN_STUDY: bool = True
RESUME_STUDY: bool = False

ORDERS: tuple[int, ...] = (0, 1, 2)
TRAINING_SEEDS: tuple[int, ...] = (42,)
EPOCHS: int = 60
STEPS_PER_EPOCH: int | None = None
BATCH_SIZE: int = 8
DATA_MULTIPLIER: int = 3
PATCH_SIZE: tuple[int, int, int] = (80, 80, 80)
FILTERS: tuple[int, int, int, int, int] = (16, 32, 64, 128, 256)
INFERENCE_STRIDE: int = 20
SAVE_PREDICTIONS: bool = False

# False: no validation pass, preset LR schedule, final epoch for evaluation.
USE_VALIDATION_FOR_SELECTION: bool = False
INITIAL_LEARNING_RATE: float = 0.001
MINIMUM_LEARNING_RATE: float = 1e-7

fixed_epoch_training: bool = not USE_VALIDATION_FOR_SELECTION
checkpoint_monitor: str = "loss"
if USE_VALIDATION_FOR_SELECTION:
    checkpoint_monitor = "val_loss"

training = TrainingConfig(
    model_name="d1_interpolation", epochs=EPOCHS, batch_size=BATCH_SIZE,
    steps_per_epoch=STEPS_PER_EPOCH, data_multiplier=DATA_MULTIPLIER,
    seed=42, output_dir=str(REPO_ROOT / "results"),
    monitor=checkpoint_monitor, monitor_mode="min", fixed_epochs=fixed_epoch_training, default_callbacks=False,
    fit_kwargs={"verbose": 1},
)
preprocessing = PreprocessingConfig(
    reference=REFERENCE, patch_size=PATCH_SIZE, spacing_mm=(0.1, 0.1, 0.1),
    correct_units=True, augment=True, random_crop=True, crop_mode="center",
    resampling_order=1, intensity_augmentation_grid="native",
    flip_probability=0.5,
)
postprocessing = PostprocessingConfig(
    clean_brain_mask=True,
    fill_small_holes=True, holes_max_area=20000,
    remove_small_objects=True, object_min_area=30000,
    split_ventricles=True, use_centroids=True,
    default_hemisphere="right", lesion_voxels_threshold=10,
)
inference = InferenceConfig(
    enabled=True, stride=INFERENCE_STRIDE, save_predictions=SAVE_PREDICTIONS,
    original_label_ids=True, postprocessing=postprocessing,
)
config = ExperimentConfig(
    data=DataConfig(root=DATA_ROOT, datasets=(DatasetSpec(name="D1"),)),
    preprocessing=preprocessing, training=training,
    labels=LabelConfig(unknown="background"),
    model=ModelConfig(
        architecture="3drnet", filters=FILTERS,
        residual=True, attention=True, max_pooling=True, classifier=False,
    ),
    loss=LossConfig(regions="dice_ce", gamma=0.8, brain_alpha=0.6, batch_wise=True),
    inference=inference, initialization=InitializationConfig(),
    fixed_split=FixedSplitConfig(validation_fraction=0.2, test_fraction=0.1),
    evaluation_patches=EvaluationPatchConfig(
        patches_per_scan=3, sampling="center_biased", seed=42, left_right_pairs=False,
    ),
    learning_rate=INITIAL_LEARNING_RATE, split_seed=42,
)
config.validate()

## 2. Configure callbacks directly

The pilot uses a preset cosine learning-rate schedule and fixed epoch budget so every model receives
the same number of optimizer steps. Logging callbacks are visible below. Add
adaptive callbacks here only as an explicit shared protocol change in a fresh
study. Best validation weights and recovery checkpoints are saved by the core;
each fit receives fresh callback instances.

In [15]:
import tensorflow as tf
from src.experiments.schedules import CosineEpochSchedule

# Select the policy explicitly; test/holdout metrics never drive these callbacks.
callbacks: list[tf.keras.callbacks.Callback] = []
if USE_VALIDATION_FOR_SELECTION:
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", mode="min", factor=0.5, patience=10,
        min_lr=MINIMUM_LEARNING_RATE, verbose=1,
    )
    callbacks.append(reduce_lr)
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", mode="min", patience=20, verbose=1,
        restore_best_weights=False,
    )
    callbacks.append(early_stopping)
else:
    schedule = CosineEpochSchedule(
        initial_learning_rate=config.learning_rate,
        minimum_learning_rate=MINIMUM_LEARNING_RATE,
        epochs=config.training.epochs,
    )
    scheduler = tf.keras.callbacks.LearningRateScheduler(schedule=schedule, verbose=1)
    callbacks.append(scheduler)

csv_filename: str = "training.csv"
csv_logger = tf.keras.callbacks.CSVLogger(filename=csv_filename, append=True)
callbacks.append(csv_logger)
log_dir: str = "tensorboard"
tensorboard = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=0)
callbacks.append(tensorboard)


In [16]:
prepared = prepare_data(config)
selection = study_selection(
    data=prepared, mode="fixed_split", study_dir=STUDY_DIR,
    split_file=SPLIT_FILE, resume=RESUME_STUDY,
    reference_file=REFERENCE_SPLITS, reference_fold=REFERENCE_FOLD,
)
split = selection.split
display(selection.summary)
display(prepared.annotation_summary())
print("Study:", STUDY_DIR)
print("Fits:", len(ORDERS) * len(TRAINING_SEEDS))

,Role,Animals,AnimalFraction,Scans
0,train,71,0.633929,71
1,validation,18,0.160714,18
2,test,23,0.205357,23


,Dataset,ClassIndex,Class,AnnotationMeaning,SourceLabels,AnnotatedScans,UnannotatedScans,TotalScans
0,D1,0,Background,task remainder; background supervision is loss...,[0],112,0,112
1,D1,1,Lesion,"full-volume ROI, including known negatives",[1],112,0,112
2,D1,2,Lateral ventricles,"full-volume ROI, including known negatives","[3, 13]",112,0,112
3,D1,3,Third ventricle,"full-volume ROI, including known negatives",[21],112,0,112


Study: c:\Users\MDesalvo\Desktop\RNET3D\results\interpolation\d1_orders_0_1_2_reference_fold_0_balanced
Fits: 3


## 3. Check which inputs actually require resampling

Reorientation and unit correction precede resampling. Scans already at the
target spacing bypass resampling, so changing spline order cannot alter their
input at this step. The audit below uses the actual production prefix and
reports this by split and sequence. Inspect T2w separately instead of assuming
every D1 scan is affected. The training cell refuses a completely unaffected
inventory; no artificial displacement or resolution change is introduced.
The flag follows the resampler's exact tolerance: tiny header rounding can mark
nominally isotropic scans as affected. Inspect the actual spacing and relative
change rather than treating every flagged scan as a large resolution change.

In [17]:
base_pipeline = make_preprocessor(config=config, stage="inference")
resample_index = next(index for index, step in enumerate(base_pipeline.steps) if isinstance(step, Resample))
prefix_steps = base_pipeline.steps[:resample_index]
prefix = Preprocessor(steps=prefix_steps)

roles: dict[str, str] = {}
for role, animals in (("train", split.train), ("validation", split.validation), ("test", split.test)):
    for animal in animals:
        roles[animal] = role

audit_rows: list[dict[str, object]] = []
for key in prepared.loader.get_subjects_list():
    scan = prepared.loader.get_subject(key)
    subject = prepared.subjects.subject_for_scan(key)
    corrected = prefix.preprocess(imgs=scan.image)
    spacing = nib.affines.voxel_sizes(corrected.affine)
    needs_resampling = not np.allclose(spacing, config.preprocessing.spacing_mm, rtol=0, atol=1e-8)
    target_spacing = np.asarray(config.preprocessing.spacing_mm)
    relative_change = np.abs(spacing / target_spacing - 1)
    maximum_change = float(relative_change.max())
    anisotropy = float(spacing.max() / spacing.min())
    modality = scan.metadata.get("modality", scan.modality)
    audit_rows.append({
        "Subject": subject, "Scan": scan.id, "Dataset": scan.subject_group[0],
        "DatasetRole": roles[subject], "Modality": modality,
        "SpacingBeforeResampling": tuple(float(value) for value in spacing),
        "MaxRelativeSpacingChange": maximum_change, "SpacingAnisotropy": anisotropy,
        "ResamplingNeeded": needs_resampling,
    })

grid_audit = pd.DataFrame(audit_rows)
grid_counts = grid_audit.groupby(["DatasetRole", "Modality", "ResamplingNeeded"]).agg(
    Animals=("Subject", "nunique"), Scans=("Scan", "size"),
).reset_index()
display(grid_counts)
display(grid_audit)

,DatasetRole,Modality,ResamplingNeeded,Animals,Scans
0,test,T1w,True,17,17
1,test,T2w,True,6,6
2,train,T1w,True,55,55
3,train,T2w,True,16,16
4,validation,T1w,True,14,14
5,validation,T2w,True,4,4


,Subject,Scan,Dataset,DatasetRole,Modality,SpacingBeforeResampling,MaxRelativeSpacingChange,SpacingAnisotropy,ResamplingNeeded
0,D1/SUBJ_0001,SUBJ_0001,D1,test,T1w,"(0.0999980723429269, 0.09999905685646354, 0.10...",0.000054,1.000073,True
1,D1/SUBJ_0002,SUBJ_0002,D1,train,T1w,"(0.09999374931515266, 0.10000209683871568, 0.0...",0.000067,1.000088,True
2,D1/SUBJ_0003,SUBJ_0003,D1,test,T1w,"(0.10000694374995052, 0.09999392145956543, 0.0...",0.000069,1.000130,True
3,D1/SUBJ_0004,SUBJ_0004,D1,test,T1w,"(0.10000655977523723, 0.10000672290783372, 0.1...",0.000067,1.000067,True
4,D1/SUBJ_0005,SUBJ_0005,D1,train,T1w,"(0.09999356199845971, 0.10000600198176289, 0.1...",0.000064,1.000124,True
...,...,...,...,...,...,...,...,...,...
107,D1/SUBJ_0108,SUBJ_0108,D1,train,T2w,"(0.10000000149622051, 0.10000000000000035, 0.2...",2.000000,3.000000,True
108,D1/SUBJ_0109,SUBJ_0109,D1,train,T2w,"(0.10000000149622051, 0.10000000000000035, 0.2...",2.000000,3.000000,True
109,D1/SUBJ_0110,SUBJ_0110,D1,test,T2w,"(0.1, 0.10000000000000007, 0.30000000000000016)",2.000000,3.000000,True
110,D1/SUBJ_0111,SUBJ_0111,D1,train,T2w,"(0.1, 0.10000000000000007, 0.30000000000000016)",2.000000,3.000000,True


In [18]:
@dataclass(frozen=True)
class InterpolationVariant:
    """One MRI resampling order; every other experiment setting is shared."""

    name: str
    order: int

    def configuration(self, base: ExperimentConfig) -> ExperimentConfig:
        """Store the order in the normal experiment/recovery configuration."""
        settings = replace(base.preprocessing, resampling_order=self.order)
        configured = replace(base, preprocessing=settings)
        configured.validate()
        return configured


if len(set(ORDERS)) != len(ORDERS) or 1 not in ORDERS:
    raise ValueError("Choose distinct interpolation orders and retain order 1 as reference")

variants: list[InterpolationVariant] = []
plan_rows: list[dict[str, object]] = []
for order in ORDERS:
    name = f"order_{order}"
    variant = InterpolationVariant(name=name, order=order)
    variant.configuration(config)
    variants.append(variant)
    plan_rows.append({
        "Variant": name, "MRIResamplingOrder": order,
        "MaskOrder": 0, "AffineMRI": "linear", "Epochs": EPOCHS,
        "StepsPerEpoch": STEPS_PER_EPOCH, "Initialization": "scratch",
    })

plan = pd.DataFrame(plan_rows)
display(plan)

,Variant,MRIResamplingOrder,MaskOrder,AffineMRI,Epochs,StepsPerEpoch,Initialization
0,order_0,0,0,linear,40,None,scratch
1,order_1,1,0,linear,40,None,scratch
2,order_2,2,0,linear,40,None,scratch


## 4. Run or resume the four models

Set `RUN_STUDY=True` in configuration and run the following cell. To resume an
interruption, keep the same study directory/settings, set `RESUME_STUDY=True`
and rerun the preceding cells. Completed fits are reused. Changing orders,
epochs, steps or seeds requires a fresh study directory.

The same saved evaluation coordinates are reused, with each model's selected
MRI order and identical nearest-neighbour GT. Validation selects best weights;
full-volume validation/test metrics are then measured once per evaluation pass.
No training starts with the default switch.

In [19]:
study = None
if RUN_STUDY:
    if not grid_audit["ResamplingNeeded"].any():
        raise ValueError("No input requires target resampling; these orders would produce identical inputs")
    train_generator = prepared.make_generator(split.train, training=True)
    if STEPS_PER_EPOCH is not None and STEPS_PER_EPOCH > len(train_generator):
        raise ValueError("STEPS_PER_EPOCH exceeds the available training batches; reduce it or use None")
    del train_generator

    study = run_study(
        data=prepared, variants=variants, plan=plan, study_dir=STUDY_DIR,
        mode="fixed_split", split=split, seeds=TRAINING_SEEDS,
        resume=RESUME_STUDY, callbacks=callbacks,
    )
    grid_audit.to_csv(STUDY_DIR / "resampling_audit.csv", index=False)
    print("Study saved to:", study.study_dir)
else:
    print("Review the configuration and grid audit, then set RUN_STUDY=True.")

Generator configuration:
- Dataset path: D:\Other files\TESI_CNN\MANUSCRIPT_DATASET\normalized
- Dimensions: (80, 80, 80)
- Batch size: 8
- Number of input channels: 1
- Number of classes: 4
- Shuffle: True

Generator configuration:
- Dataset path: D:\Other files\TESI_CNN\MANUSCRIPT_DATASET\normalized
- Dimensions: (80, 80, 80)
- Batch size: 8
- Number of input channels: 1
- Number of classes: 4
- Shuffle: True

Generator configuration:
- Dataset path: D:\Other files\TESI_CNN\MANUSCRIPT_DATASET\normalized
- Dimensions: (80, 80, 80)
- Batch size: 8
- Number of input channels: 1
- Number of classes: 4
- Shuffle: False

Training checkpoints: C:\Users\MDesalvo\Desktop\RNET3D\results\interpolation\d1_orders_0_1_2_reference_fold_0_balanced\runs\order_0\seed_42\fold_0
Epoch 1/40


KeyboardInterrupt: 

In [ ]:
CURVE_METRIC: str = "loss"  # Also try regions_mean_dice after inspecting available metrics.

history_rows: list[dict[str, object]] = []
if study is not None:
    for run in study.runs.values():
        variant_name = run.run_dir.parent.name
        saved_config = ExperimentConfig.load(run.run_dir / "experiment.json")
        for fold, history in enumerate(run.histories):
            for metric, values in history.items():
                for epoch, value in enumerate(values, start=1):
                    if value is not None:
                        history_rows.append({
                            "Variant": variant_name, "Seed": str(saved_config.training.seed),
                            "Fold": fold, "Epoch": epoch, "Metric": metric, "Value": value,
                        })
    histories = pd.DataFrame(history_rows)
    print("Available metrics:", sorted(histories["Metric"].unique()))
    curve_names = [CURVE_METRIC, f"val_{CURVE_METRIC}"]
    curve_rows = histories.loc[histories["Metric"].isin(curve_names)]
    if not curve_rows.empty:
        curve_figure = px.line(
            curve_rows, x="Epoch", y="Value", color="Variant", line_dash="Seed",
            facet_row="Metric", title="Matched short-budget training curves",
        )
        curve_figure.show(renderer="vscode")

Available metrics: ['brain_mask_loss', 'brain_mask_mean_dice', 'loss', 'regions_loss', 'regions_mean_dice', 'val_brain_mask_loss', 'val_brain_mask_mean_dice', 'val_loss', 'val_regions_loss', 'val_regions_mean_dice']


## 5. Compare validation and test measurements

This section reads saved results, so it also works after a restart with the
setup/configuration cells and an explicit `RESULTS_DIR`; no fitting is required.
Validation lesion Dice supplies the exploratory ranking. Full-volume reports
include postprocessed lesion, separate ventricles and brain metrics: Dice, IoU,
HD95, ASSD, sensitivity, precision and volume errors. Native deterministic patch
metrics remain separate. Paired test comparisons use **order 1 as reference**;
their differences are candidate minus linear, with animal intervals and Holm
correction. Inspect sample counts and empty-mask statuses.

One split and one seed do not establish a universally better interpolation
method. A short-budget difference may reflect convergence speed. Do not select
settings from the test ranking and then describe that cohort as untouched.

In [ ]:
from src.evaluation.paper_tables import PaperTableConfig, paper_tables_from_folder

RESULTS_DIR: Path = STUDY_DIR
BOOTSTRAP_SAMPLES: int = 1000

validation_tables = None
test_tables = None
metric_file = RESULTS_DIR / "full_volume_metrics.csv"
if metric_file.is_file() and metric_file.stat().st_size > 10:
    test_settings = PaperTableConfig(
        roles=("test",), reference_variant="order_1", bootstrap_samples=BOOTSTRAP_SAMPLES,
    )
    test_tables = paper_tables_from_folder(RESULTS_DIR, settings=test_settings)

    if USE_VALIDATION_FOR_SELECTION:
        validation_settings = PaperTableConfig(
            roles=("validation",), reference_variant="order_1", bootstrap_samples=BOOTSTRAP_SAMPLES,
        )
        validation_tables = paper_tables_from_folder(RESULTS_DIR, settings=validation_settings)

        validation_scores = validation_tables.statistics
        lesion_dice = validation_scores.loc[
            validation_scores["Head"].eq("regions") & validation_scores["Region"].eq("Lesion")
            & validation_scores["Metric"].eq("Dice")
        ]
        ranking_columns = ["Variant", "Seed", "Mean", "Animals", "AnimalCILower", "AnimalCIUpper"]
        validation_ranking = lesion_dice[ranking_columns].sort_values(["Seed", "Mean"], ascending=[True, False])
        print("Exploratory validation lesion-Dice ranking:")
        display(validation_ranking)
        print("Native validation patch measurements:")
        display(validation_tables.patches)
    else:
        print("No validation ranking: epochs and LR schedule were fixed before testing.")
    print("Held-out test metrics and anatomy/brain averages:")
    display(test_tables.fold_coverage)
    display(test_tables.segmentation)
    display(test_tables.region_sets)
    display(test_tables.animal_intervals)
    display(test_tables.patches)
    display(test_tables.comparisons)
    display(test_tables.costs)
else:
    print("No saved study measurements yet. Run the study or select an explicit RESULTS_DIR.")

c:\Users\MDesalvo\AppData\Local\miniconda3\envs\rnet\lib\site-packages\scipy\stats\_morestats.py:4088: UserWarning:

Exact p-value calculation does not work if there are zeros. Switching to normal approximation.

c:\Users\MDesalvo\AppData\Local\miniconda3\envs\rnet\lib\site-packages\scipy\stats\_morestats.py:4088: UserWarning:

Exact p-value calculation does not work if there are zeros. Switching to normal approximation.

c:\Users\MDesalvo\AppData\Local\miniconda3\envs\rnet\lib\site-packages\scipy\stats\_morestats.py:4088: UserWarning:

Exact p-value calculation does not work if there are zeros. Switching to normal approximation.

c:\Users\MDesalvo\AppData\Local\miniconda3\envs\rnet\lib\site-packages\scipy\stats\_morestats.py:4102: UserWarning:

Sample size too small for normal approximation.

c:\Users\MDesalvo\AppData\Local\miniconda3\envs\rnet\lib\site-packages\scipy\stats\_morestats.py:4088: UserWarning:

Exact p-value calculation does not work if there are zeros. Switching to norma

Exploratory validation lesion-Dice ranking:


,Variant,Seed,Mean,Animals,AnimalCILower,AnimalCIUpper
132,order_2,42,0.034208,23,0.027950,0.040737
32,order_0,42,0.003425,23,0.002826,0.004091
82,order_1,42,0.003164,23,0.002718,0.003652


Native validation patch measurements:


,Variant,Seed,DatasetRole,Dataset,Head,Region,Metric,Unit,Mean,FoldSD,FiniteFolds,Folds
0,order_0,42,validation,D1,brain_mask,Brain,brain_mask_loss,unitless,0.796065,NaN,1,1
1,order_0,42,validation,D1,brain_mask,Brain,brain_mask_mean_dice,unitless,0.724657,NaN,1,1
2,order_0,42,validation,D1,regions,Regions,regions_loss,unitless,0.594370,NaN,1,1
3,order_0,42,validation,D1,regions,Regions,regions_mean_dice,unitless,0.008798,NaN,1,1
4,order_0,42,validation,D1,total,All heads,loss,unitless,1.472904,NaN,1,1
5,order_1,42,validation,D1,brain_mask,Brain,brain_mask_loss,unitless,0.749802,NaN,1,1
6,order_1,42,validation,D1,brain_mask,Brain,brain_mask_mean_dice,unitless,0.734057,NaN,1,1
7,order_1,42,validation,D1,regions,Regions,regions_loss,unitless,0.735922,NaN,1,1
8,order_1,42,validation,D1,regions,Regions,regions_mean_dice,unitless,0.009482,NaN,1,1
9,order_1,42,validation,D1,total,All heads,loss,unitless,1.568199,NaN,1,1


...

,Variant,Seed,DatasetRole,Dataset,Condition,Head,Region,Folds,ObservedAnimals,Animals,DiceFolds,BoundaryAnimals,Dice,HD95 (mm),ASSD (mm),AVE (mm3),RVE (%),Sensitivity,Precision,IoU
0,order_0,42,test,D1,TBI,region_set,all_roi,1,12,12,1,12,0.002,12.478,3.553,277.486,2454.314,0.027,0.001,0.001
1,order_0,42,test,D1,TBI,region_set,all_roi_and_brain,1,12,12,1,12,0.114,10.810,3.116,460.533,1979.884,0.224,0.079,0.079
2,order_0,42,test,D1,TBI,region_set,anatomy_and_brain,1,12,12,1,12,0.141,9.412,2.766,319.900,1673.147,0.265,0.098,0.098
3,order_0,42,test,D1,TBI,region_set,anatomy_only,1,12,12,1,12,0.001,11.170,3.234,28.960,2209.853,0.012,0.001,0.001
4,order_1,42,test,D1,TBI,region_set,all_roi,1,12,12,1,12,0.002,16.472,4.145,454.254,3753.351,0.040,0.001,0.001
5,order_1,42,test,D1,TBI,region_set,all_roi_and_brain,1,12,12,1,12,0.117,14.185,3.593,580.795,3006.341,0.234,0.082,0.082
6,order_1,42,test,D1,TBI,region_set,anatomy_and_brain,1,12,12,1,12,0.145,13.665,3.359,300.823,2390.050,0.267,0.102,0.102
7,order_1,42,test,D1,TBI,region_set,anatomy_only,1,12,12,1,12,0.002,16.570,4.022,38.777,3191.042,0.018,0.001,0.001
8,order_2,42,test,D1,TBI,region_set,all_roi,1,12,12,1,12,0.016,11.861,2.701,327.895,3105.001,0.297,0.010,0.008
9,order_2,42,test,D1,TBI,region_set,all_roi_and_brain,1,12,12,1,12,0.122,10.305,2.441,532.462,2478.635,0.440,0.084,0.082


,Variant,Seed,DatasetRole,Dataset,Condition,Head,Region,Folds,ObservedAnimals,Animals,...,Dice,HD95 (mm),ASSD (mm),AVE (mm3),RVE (%),Sensitivity,Precision,IoU,ReferenceVolume (mm3),PredictionVolume (mm3)
0,order_0,42,test,D1,TBI,brain_mask,Brain,1,12,12,...,"0.560 [0.550, 0.570]","4.287 [4.108, 4.497]","1.417 [1.355, 1.493]","1192.720 [717.957, 2114.952]","155.725 [149.256, 162.384]","0.996 [0.993, 0.997]","0.390 [0.381, 0.400]","0.389 [0.380, 0.399]","770.893 [456.725, 1370.943]","1963.613 [1180.950, 3484.945]"
1,order_0,42,test,D1,TBI,region_set,all_roi,1,12,12,...,"0.002 [0.002, 0.003]","12.478 [9.554, 17.786]","3.553 [3.030, 4.425]","277.486 [159.019, 507.907]","2454.314 [1688.024, 3678.691]","0.027 [0.022, 0.031]","0.001 [0.001, 0.001]","0.001 [0.001, 0.001]",NaN,NaN
2,order_0,42,test,D1,TBI,region_set,all_roi_and_brain,1,12,12,...,"0.114 [0.112, 0.116]","10.810 [8.451, 15.113]","3.116 [2.696, 3.831]","460.533 [273.516, 829.141]","1979.884 [1380.281, 2943.984]","0.224 [0.217, 0.234]","0.079 [0.077, 0.081]","0.079 [0.077, 0.081]",NaN,NaN
3,order_0,42,test,D1,TBI,region_set,anatomy_and_brain,1,12,12,...,"0.141 [0.139, 0.144]","9.412 [7.054, 13.822]","2.766 [2.301, 3.525]","319.900 [194.222, 564.495]","1673.147 [951.890, 2869.927]","0.265 [0.257, 0.277]","0.098 [0.096, 0.100]","0.098 [0.096, 0.100]",NaN,NaN
4,order_0,42,test,D1,TBI,region_set,anatomy_only,1,12,12,...,"0.001 [0.001, 0.002]","11.170 [8.049, 16.982]","3.234 [2.616, 4.225]","28.960 [17.807, 47.746]","2209.853 [1216.891, 3837.176]","0.012 [0.009, 0.015]","0.001 [0.001, 0.001]","0.001 [0.000, 0.001]",NaN,NaN
5,order_0,42,test,D1,TBI,regions,Contra ventricle,1,12,12,...,"0.001 [0.001, 0.002]","9.619 [7.256, 13.839]","3.133 [2.511, 4.020]","16.547 [11.157, 24.055]","900.579 [608.459, 1246.746]","0.007 [0.005, 0.011]","0.001 [0.001, 0.001]","0.001 [0.000, 0.001]","4.427 [1.385, 9.930]","20.974 [12.734, 34.011]"
6,order_0,42,test,D1,TBI,regions,Ipsi ventricle,1,12,12,...,"0.003 [0.002, 0.004]","12.708 [8.124, 21.341]","3.106 [2.190, 4.723]","30.163 [19.837, 47.317]","4221.982 [1538.332, 9210.885]","0.030 [0.021, 0.039]","0.002 [0.001, 0.002]","0.001 [0.001, 0.002]","1.818 [0.787, 3.334]","31.980 [20.851, 50.547]"
7,order_0,42,test,D1,TBI,regions,Lesion,1,12,12,...,"0.004 [0.003, 0.005]","16.239 [13.818, 20.336]","4.480 [4.088, 5.044]","1023.065 [581.642, 1888.017]","3171.528 [2810.405, 3571.352]","0.069 [0.057, 0.081]","0.002 [0.002, 0.003]","0.002 [0.002, 0.003]","36.871 [17.566, 72.773]","1059.936 [600.462, 1960.914]"
8,order_0,42,test,D1,TBI,regions,Third ventricle,1,12,12,...,"0.000 [0.000, 0.000]","11.480 [8.625, 16.785]","3.454 [3.048, 4.151]","40.169 [21.583, 72.245]","1569.210 [1111.021, 2147.585]","0.001 [0.000, 0.002]","0.000 [0.000, 0.000]","0.000 [0.000, 0.000]","3.625 [1.540, 7.399]","43.794 [23.353, 79.647]"
9,order_1,42,test,D1,TBI,brain_mask,Brain,1,12,12,...,"0.575 [0.564, 0.585]","5.271 [4.945, 5.550]","1.449 [1.403, 1.502]","1086.959 [666.864, 1898.939]","144.481 [137.725, 151.459]","0.989 [0.985, 0.993]","0.406 [0.395, 0.416]","0.404 [0.393, 0.414]","770.893 [456.725, 1370.943]","1857.852 [1130.598, 3267.373]"


,Variant,Seed,DatasetRole,Dataset,Head,Region,Metric,Unit,Mean,FoldSD,FiniteFolds,Folds
0,order_0,42,test,D1,brain_mask,Brain,brain_mask_loss,unitless,0.888812,NaN,1,1
1,order_0,42,test,D1,brain_mask,Brain,brain_mask_mean_dice,unitless,0.677280,NaN,1,1
2,order_0,42,test,D1,regions,Regions,regions_loss,unitless,0.590497,NaN,1,1
3,order_0,42,test,D1,regions,Regions,regions_mean_dice,unitless,0.007140,NaN,1,1
4,order_0,42,test,D1,total,All heads,loss,unitless,1.561779,NaN,1,1
5,order_1,42,test,D1,brain_mask,Brain,brain_mask_loss,unitless,0.839369,NaN,1,1
6,order_1,42,test,D1,brain_mask,Brain,brain_mask_mean_dice,unitless,0.687751,NaN,1,1
7,order_1,42,test,D1,regions,Regions,regions_loss,unitless,0.726279,NaN,1,1
8,order_1,42,test,D1,regions,Regions,regions_mean_dice,unitless,0.008448,NaN,1,1
9,order_1,42,test,D1,total,All heads,loss,unitless,1.648124,NaN,1,1


,DatasetRole,Dataset,Region,Metric,Unit,Reference,Candidate,PairedAnimals,MeanDifference,PairedScans,CILower,CIUpper,WilcoxonP,Status,HolmP,Seed,Condition,Head
0,test,D1,Brain,ASSD,mm,order_1,order_0,12,-0.032950,12,-0.071813,0.013044,0.233398,ok,1.000000,42,TBI,brain_mask
1,test,D1,Brain,AVE,mm3,order_1,order_0,12,105.760666,12,47.490198,217.059462,0.000488,ok,0.039062,42,TBI,brain_mask
2,test,D1,Brain,Dice,fraction,order_1,order_0,12,-0.014805,12,-0.017242,-0.012270,0.000488,ok,0.039062,42,TBI,brain_mask
3,test,D1,Brain,HD95,mm,order_1,order_0,12,-0.983937,12,-1.284730,-0.660673,0.002441,ok,0.078125,42,TBI,brain_mask
4,test,D1,Brain,IoU,fraction,order_1,order_0,12,-0.014463,12,-0.016862,-0.011860,0.000488,ok,0.039062,42,TBI,brain_mask
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,test,D1,Third ventricle,HD95,mm,order_1,order_2,12,-3.228643,12,-3.520671,-2.983296,0.000488,ok,0.039062,42,TBI,regions
76,test,D1,Third ventricle,IoU,fraction,order_1,order_2,12,0.007034,12,0.005474,0.008904,0.000488,ok,0.039062,42,TBI,regions
77,test,D1,Third ventricle,Precision,fraction,order_1,order_2,12,0.007106,12,0.005527,0.009006,0.000488,ok,0.039062,42,TBI,regions
78,test,D1,Third ventricle,RVE,%,order_1,order_2,12,4605.698848,12,3096.513265,6509.935276,0.000488,ok,0.039062,42,TBI,regions


,Variant,Kind,DatasetRole,Dataset,Device,Precision,ProvenanceStatus,Metric,Unit,Recorded,Rows,Mean,Median,Minimum,Maximum
0,order_0,training,training,unknown,unknown,unknown,missing_hardware_metadata,best_epoch,count,1,1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,order_0,training,training,unknown,unknown,unknown,missing_hardware_metadata,epochs_completed,count,1,1,1.000000e+01,1.000000e+01,1.000000e+01,1.000000e+01
2,order_0,training,training,unknown,unknown,unknown,missing_hardware_metadata,evaluation_seconds,s,1,1,1.491324e+03,1.491324e+03,1.491324e+03,1.491324e+03
3,order_0,training,training,unknown,unknown,unknown,missing_hardware_metadata,parameter_count,count,1,1,7.173518e+06,7.173518e+06,7.173518e+06,7.173518e+06
4,order_0,training,training,unknown,unknown,unknown,missing_hardware_metadata,training_seconds,s,1,1,1.075879e+03,1.075879e+03,1.075879e+03,1.075879e+03
5,order_1,training,training,unknown,unknown,unknown,missing_hardware_metadata,best_epoch,count,1,1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
6,order_1,training,training,unknown,unknown,unknown,missing_hardware_metadata,epochs_completed,count,1,1,1.000000e+01,1.000000e+01,1.000000e+01,1.000000e+01
7,order_1,training,training,unknown,unknown,unknown,missing_hardware_metadata,evaluation_seconds,s,1,1,1.876557e+03,1.876557e+03,1.876557e+03,1.876557e+03
8,order_1,training,training,unknown,unknown,unknown,missing_hardware_metadata,parameter_count,count,1,1,7.173518e+06,7.173518e+06,7.173518e+06,7.173518e+06
9,order_1,training,training,unknown,unknown,unknown,missing_hardware_metadata,training_seconds,s,1,1,1.122720e+03,1.122720e+03,1.122720e+03,1.122720e+03


## 6. Separate T1w/T2w and affected/unaffected scans

The tables below average scans within animals and keep seeds separate. They
show Dice by MRI sequence and whether target resampling actually occurred.
Inspect the individual points and counts: a pooled D1 average can dilute an
effect confined to anisotropic T2w inputs. Empty-reference Dice conventions
remain visible in the saved full-volume CSV, alongside boundary statuses.

In [ ]:
if test_tables is not None:
    measurements = pd.read_csv(metric_file)
    audit_file = RESULTS_DIR / "resampling_audit.csv"
    if not audit_file.is_file():
        raise FileNotFoundError("Select the study's saved resampling_audit.csv for sequence-specific interpretation")
    saved_audit = pd.read_csv(audit_file)
    join_keys = ["Subject", "Scan", "Dataset"]
    audit_columns = [*join_keys, "ResamplingNeeded"]
    measurements = measurements.merge(saved_audit[audit_columns], on=join_keys, how="left", validate="many_to_one")
    if measurements["ResamplingNeeded"].isna().any():
        raise ValueError("Some measured scans are missing from the saved grid audit")

    test_dice = measurements.loc[measurements["DatasetRole"].eq("test") & measurements["Metric"].eq("Dice")].copy()
    grouping = ["Variant", "Seed", "Modality", "ResamplingNeeded", "Head", "Region"]
    animal_dice = test_dice.groupby([*grouping, "Subject"], dropna=False)["Value"].mean().reset_index()
    sequence_summary = animal_dice.groupby(grouping, dropna=False).agg(
        Animals=("Value", "count"), MeanDice=("Value", "mean"), SDDice=("Value", "std"),
    ).reset_index()
    display(sequence_summary)
    sequence_figure = px.box(
        animal_dice, x="Variant", y="Value", color="Modality", facet_col="Region",
        facet_col_wrap=3, points="all", hover_data=["Subject", "Seed", "ResamplingNeeded"],
        title="Test animal Dice by sequence (descriptive; empty-mask conventions retained)",
        labels={"Value": "Animal Dice"},
    )
    sequence_figure.show(renderer="vscode")

,Variant,Seed,Modality,ResamplingNeeded,Head,Region,Animals,MeanDice,SDDice
0,order_0,42,T1w,True,brain_mask,Brain,10,0.562824,0.017310
1,order_0,42,T1w,True,regions,Contra ventricle,10,0.001634,0.001047
2,order_0,42,T1w,True,regions,Ipsi ventricle,10,0.003255,0.002544
3,order_0,42,T1w,True,regions,Lesion,10,0.004508,0.001506
4,order_0,42,T1w,True,regions,Third ventricle,10,0.000028,0.000050
5,order_0,42,T2w,True,brain_mask,Brain,2,0.547444,0.022570
6,order_0,42,T2w,True,regions,Contra ventricle,2,0.000411,0.000582
7,order_0,42,T2w,True,regions,Ipsi ventricle,2,0.000879,0.001243
8,order_0,42,T2w,True,regions,Lesion,2,0.002708,0.001184
9,order_0,42,T2w,True,regions,Third ventricle,2,0.000518,0.000660


In [ ]:
EXPORT_PAPER_TABLES: bool = False
PAPER_OUTPUT: Path | None = None  # Fresh explicit folder; never overwrite an existing export.

if EXPORT_PAPER_TABLES:
    if test_tables is None or PAPER_OUTPUT is None:
        raise ValueError("Load the results and select a fresh PAPER_OUTPUT folder")
    PAPER_OUTPUT.mkdir(parents=True, exist_ok=False)
    validation_output = PAPER_OUTPUT / "validation"
    test_output = PAPER_OUTPUT / "test"
    if validation_tables is not None:
        validation_tables.export(validation_output)
    test_tables.export(test_output)
    if validation_tables is not None:
        validation_ranking.to_csv(PAPER_OUTPUT / "validation_lesion_dice_ranking.csv", index=False)
    sequence_summary.to_csv(PAPER_OUTPUT / "test_sequence_summary.csv", index=False)
    print("Paper tables saved to:", PAPER_OUTPUT)